In [51]:
import json
import logging
import os
import pickle

import hydra
import numpy as np
import pytorch_lightning as pl
import torch
import yaml
from hydra.utils import instantiate
from omegaconf import DictConfig, OmegaConf
from pytorch_lightning.loggers import CSVLogger
from torch import nn
from transcriptformer.model.embedding_surgery import change_embedding_layer
from transcriptformer.tokenizer.vocab import load_vocabs_and_embeddings

TF_CFG = os.getenv(
    "TF_CFG",
    "/mnt/czi-sci-ai/project-rbio/transcriptformer/inference_config.yaml",
)
TF_MODEL_CKPT = os.getenv(
    "TF_MODEL_CKPT",
    "/mnt/czi-sci-ai/project-rbio/transcriptformer/tf_sapiens",
)
GENE2ENSEMBL_ID_FILEPATH = os.getenv(
    "GENE2ENSEMBL_ID_FILEPATH",
    "/mnt/czi-sci-ai/project-rbio/transcriptformer/gene2ensembl_ids.pkl",
)


def call_vcm(
    gene_perturbed,
    gene_monitored,
    gene2ensembl_id,
    model,
    gene_vocab,
    verification_type="gene_similarity",
):
    gene_perturbed_ensembl_id = (
        str(np.random.choice(gene2ensembl_id[gene_perturbed]))
        if gene_perturbed in gene2ensembl_id
        else "[PAD]"
    )
    gene_monitored_ensembl_id = (
        str(np.random.choice(gene2ensembl_id[gene_monitored]))
        if gene_monitored in gene2ensembl_id
        else "[PAD]"
    )
    gene_perturbed_index = (
        gene_vocab[gene_perturbed_ensembl_id]
        if gene_perturbed_ensembl_id in gene_vocab
        else gene_vocab["[PAD]"]
    )
    gene_monitored_index = (
        gene_vocab[gene_monitored_ensembl_id]
        if gene_monitored_ensembl_id in gene_vocab
        else gene_vocab["[PAD]"]
    )

    gene_perturbed_index = torch.Tensor([gene_perturbed_index]).long()
    gene_monitored_index = torch.Tensor([gene_monitored_index]).long()

    gene_embs = model.gene_embeddings.embedding
    gene_perturbed_emb = gene_embs(gene_perturbed_index)
    gene_monitored_emb = gene_embs(gene_monitored_index)
    cos = nn.CosineSimilarity(dim=1, eps=1e-6)
    gene_similarity = cos(gene_perturbed_emb, gene_monitored_emb)

    return gene_similarity[0]


def instantiate_vcm(model_type):
    if model_type == "transcriptformer":
        gene2ensemble_id = pickle.load(open(GENE2ENSEMBL_ID_FILEPATH, "rb"))

        cfg = yaml.load(open(TF_CFG, "r"), Loader=yaml.SafeLoader)
        config_path = os.path.join(cfg["model"]["checkpoint_path"], "config.json")
        with open(config_path) as f:
            config_dict = json.load(f)
        mlflow_cfg = OmegaConf.create(config_dict)

        # Merge the MLflow config with the main config
        cfg = OmegaConf.create(cfg)
        cfg = OmegaConf.merge(mlflow_cfg, cfg)

        # Set the checkpoint paths based on the unified checkpoint_path
        cfg.model.inference_config.load_checkpoint = os.path.join(
            cfg.model.checkpoint_path, "model_weights.pt"
        )
        cfg.model.data_config.aux_vocab_path = os.path.join(
            cfg.model.checkpoint_path, "vocabs"
        )
        cfg.model.data_config.esm2_mappings_path = os.path.join(
            cfg.model.checkpoint_path, "vocabs"
        )

        (gene_vocab, aux_vocab), emb_matrix = load_vocabs_and_embeddings(cfg)
        # print('Gene VOCAB', gene_vocab)

        # Instantiate the model
        logging.info("Instantiating the model")
        model = instantiate(
            cfg.model,
            gene_vocab_dict=gene_vocab,
            aux_vocab_dict=aux_vocab,
            emb_matrix=emb_matrix,
        )
        model.eval()
        logging.info("Model instantiated successfully")

        # Check if checkpoint is supplied
        if (
            not hasattr(cfg.model.inference_config, "load_checkpoint")
            or not cfg.model.inference_config.load_checkpoint
        ):
            raise ValueError(
                "No checkpoint provided for inference. Please specify a checkpoint path in "
                "model.inference_config.load_checkpoint"
            )

        logging.info("Loading model checkpoint")
        # Instead of loading full checkpoint, just load weights
        state_dict = torch.load(
            cfg.model.inference_config.load_checkpoint, weights_only=True
        )

        # Validate and load weights
        # converter.validate_loaded_weights(model, state_dict)
        model.load_state_dict(state_dict)
        logging.info("Model weights loaded successfully")

        # Perform embedding surgery if specified in config
        if cfg.model.inference_config.pretrained_embedding is not None:
            logging.info("Performing embedding surgery")
            # Check if pretrained_embedding_paths is a list, if not convert it to a list
            if not isinstance(cfg.model.inference_config.pretrained_embedding, list):
                pretrained_embedding_paths = [
                    cfg.model.inference_config.pretrained_embedding
                ]
            else:
                pretrained_embedding_paths = (
                    cfg.model.inference_config.pretrained_embedding
                )
            model, gene_vocab = change_embedding_layer(
                model, pretrained_embedding_paths
            )
        return model, gene_vocab, gene2ensemble_id

In [52]:
model, gene_vocab, gene2ensemble_id = instantiate_vcm("transcriptformer")

2025-05-21 16:47:01,150 - INFO - Loading vocabulary file: /opt/jupyter-envs/rbio/rbio-dev-ana/work/rbio/transcriptformer/checkpoints/tf_sapiens/vocabs/assay
2025-05-21 16:47:01,151 - INFO - Loading ESM2 mappings from /opt/jupyter-envs/rbio/rbio-dev-ana/work/rbio/transcriptformer/checkpoints/tf_sapiens/vocabs
2025-05-21 16:47:02,766 - INFO - Building gene vocabulary
2025-05-21 16:47:02,994 - INFO - Instantiating the model
2025-05-21 16:47:06,301 - INFO - Model instantiated successfully
2025-05-21 16:47:06,302 - INFO - Loading model checkpoint
2025-05-21 16:47:07,055 - INFO - Model weights loaded successfully


In [53]:
gene2ensemble_id_lower = {}
for key in gene2ensemble_id.keys():
    gene2ensemble_id_lower[key.lower()] = gene2ensemble_id[key]

In [54]:
# loading a dict of cell embeddings just to have the gene names
import pickle

with open('/mnt/czi-sci-ai/project-rbio/repr/gene2vec_embeddings_filled.pkl', 'rb') as f:
    gene_emb_dict_pretrained = pickle.load(f)

In [55]:
i = 0
for key in gene_emb_dict_pretrained.keys():
    if key not in gene2ensemble_id_lower.keys():
        print(f'key {key} not found #{i}')
        i += 1

key aars not found #0
key ac118549.1 not found #1
key adprhl2 not found #2
key adss not found #3
key alg1l not found #4
key arntl not found #5
key arntl2 not found #6
key arse not found #7
key atp5md not found #8
key atp5mpl not found #9
key c11orf1 not found #10
key c11orf49 not found #11
key c11orf74 not found #12
key c11orf80 not found #13
key c12orf10 not found #14
key c12orf29 not found #15
key c12orf4 not found #16
key c12orf45 not found #17
key c12orf49 not found #18
key c12orf65 not found #19
key c12orf73 not found #20
key c15orf41 not found #21
key c16orf58 not found #22
key c16orf70 not found #23
key c16orf72 not found #24
key c16orf91 not found #25
key c17orf80 not found #26
key c18orf25 not found #27
key c19orf48 not found #28
key c19orf54 not found #29
key c1orf109 not found #30
key c1orf112 not found #31
key c20orf197 not found #32
key c20orf27 not found #33
key c4orf48 not found #34
key c5orf30 not found #35
key c5orf51 not found #36
key c7orf26 not found #37
key c9orf16

In [56]:
# replace missing keys with RANDOM embeddings
# those we have embeddings for, we just copy the embeddings in correspondence with gene name
esm_embedding_dictionary = {}

for key in gene_emb_dict_pretrained.keys():
    if key not in gene2ensemble_id_lower.keys():
        esm_embedding_dictionary[key] = np.random.randn(*model.gene_embeddings.embedding(torch.Tensor([0]).long()).shape).squeeze()
    else:
        for gene_vocab_key in gene2ensemble_id_lower[key]:
            try:
                idx = gene_vocab[gene_vocab_key]
            except KeyError:
                continue
        
            esm_embedding_dictionary[key] = model.gene_embeddings.embedding(torch.Tensor([idx]).long()).squeeze().numpy()
            break


In [57]:
with open('/mnt/czi-sci-ai/project-rbio/repr/esm_embedding_dictionary_filled.pkl', 'wb') as f:
    pickle.dump(esm_embedding_dictionary, f)